# 내 목소리로 동화 나레이션 자동 생성

1. 상단 메뉴 **런타임 > 런타임 유형 변경 > 하드웨어 가속기: GPU(T4)** 선택 후 저장
2. 셀을 위에서부터 순서대로 실행 (Shift+Enter)
3. 내 목소리 녹음 파일(10~30초, 조용한 곳, 또박또박)과 그 녹음의 정확한 대본, 그리고 동화 전체 텍스트를 업로드
4. 마지막 셀에서 완성된 나레이션을 바로 듣고, Google Drive에도 저장됨

In [ ]:
# 1) 엔진 + 이 저장소(나레이션 스크립트) 설치 (torch는 Colab GPU 런타임에 이미 CUDA 버전으로 깔려 있음)
!git clone --depth 1 https://github.com/debpalash/voicestudio /content/voicestudio
!git clone --depth 1 https://github.com/Zed-leo-23/voice-story-narrator /content/voice-story-narrator
!pip install -q transformers accelerate numpy soundfile pydub safetensors huggingface_hub sentencepiece "audioop-lts; python_version>='3.13'"

import os
import sys

sys.path.insert(0, "/content/voicestudio")
# narrate.py runs as a separate !python subprocess (step 6), so sys.path above
# only helps imports in THIS notebook kernel. The subprocess needs it via env var.
os.environ["PYTHONPATH"] = "/content/voicestudio"

import torch
print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# 2) Google Drive 연결 (결과물을 세션 종료 후에도 보존하기 위함)
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = "/content/drive/MyDrive/voice-story-narrator/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# 3) 내 목소리 레퍼런스 녹음 업로드 (wav, 10~30초, 한 사람 목소리만, 배경 잡음 없이)
from google.colab import files
uploaded = files.upload()  # my_voice.wav 같은 파일 하나 업로드
REF_AUDIO = next(iter(uploaded.keys()))
print("reference audio:", REF_AUDIO)

In [ ]:
# 4) 위 녹음의 정확한 대본을 그대로 입력 (한 글자도 틀리면 품질이 떨어짐)
REF_TEXT = "여기에 레퍼런스 녹음에서 말한 문장을 정확히 그대로 입력하세요."

In [ ]:
# 5) 동화 전체 텍스트 준비 (scenes/문단은 빈 줄로 구분)
# 먼저 샘플로 테스트하고 싶으면 이 셀 그대로, 내 동화로 바꾸고 싶으면 파일을 업로드하고 STORY_PATH를 그 파일명으로 바꾸세요
STORY_PATH = "/content/voice-story-narrator/examples/story.txt"

# 내 동화 파일을 업로드하려면 아래 두 줄의 주석을 풀고 실행
# uploaded_story = files.upload()
# STORY_PATH = next(iter(uploaded_story.keys()))

In [ ]:
# 6) 나레이션 생성
!PYTHONPATH=/content/voicestudio python /content/voice-story-narrator/narrate.py \
  --story "{STORY_PATH}" \
  --ref_audio "{REF_AUDIO}" \
  --ref_text "{REF_TEXT}" \
  --language Korean \
  --outdir "{OUTPUT_DIR}"

In [ ]:
# 7) 결과 듣기
import IPython.display as ipd
ipd.Audio(f"{OUTPUT_DIR}/full_story.wav")